# Phase 3: Existing controlled-evaluation analysis

This notebook consolidates the existing Phase 3 methodology, tables, five revised
charts, comparison interpretation, and historical chart context. It reads saved
results only: no experiment, monitor, test suite, tuning, or new research runs.
The three maintained roles are `phase3_implement.py`, `phase3_test.py`, and this
notebook; legacy entry points were retired at the US-3.32 verified cutover.
US-3.34 keeps the four user-readable outputs separate from three root-level
operational run files: the manifest, complete C0 events, and independent receipt.

**Choose `OUTPUT_DIR` below.** The default is `artifacts/phase3_outputs`; its
paired operational directory is `artifacts/phase3_operational`. Missing, stale,
or mixed-run evidence fails rather than falling back to historical outputs. For
an isolated candidate, set `PHASE3_OUTPUT_DIR` to a directory named
`phase3_outputs` before starting the kernel; its sibling `phase3_operational`
holds run-specific machine evidence. An optional
`PHASE3_REPO_ROOT` selects the corresponding complete checkout. Run all cells from
a clean kernel. Normal execution writes no files and keeps charts inline.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

root_setting = os.environ.get("PHASE3_REPO_ROOT")
if root_setting:
    ROOT = Path(root_setting).resolve()
else:
    candidates = (Path.cwd(), Path.cwd().parent)
    ROOT = next((path for path in candidates if (path / "scripts" / "phase3_implement.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Run from the repository root/scripts directory or set PHASE3_REPO_ROOT.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.phase3_implement import load_phase3_analysis_results

OUTPUT_DIR = Path(os.environ.get("PHASE3_OUTPUT_DIR", str(ROOT / "artifacts" / "phase3_outputs")))
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = ROOT / OUTPUT_DIR
OUTPUT_DIR = OUTPUT_DIR.resolve()
OPERATIONAL_DIR = (OUTPUT_DIR.with_name("phase3_operational") if OUTPUT_DIR.name == "phase3_outputs" else OUTPUT_DIR / "phase3_operational")
print(f"Repository: {ROOT}\nSaved results: {OUTPUT_DIR}\nOperational evidence: {OPERATIONAL_DIR}")

## 1. Locked experiment and validation provenance

The fixed six-sensor A/C/D/E/F/G temperature cohort uses inclusive
2022-02-24 18:50 UTC through 2022-03-10 15:53 UTC: 19,984 consecutive minute
buckets, without imputation. The threshold is an empirical stable-cohort P90,
derived by shared monitoring preparation from the six-sensor mean over this
same evaluation population. It is retrospective, not an external operating or
equipment safety limit. No Phase 1 cohort statistics row or hard-coded threshold
is authoritative; the run evidence records the derived full-precision value.
SAFE means metric < threshold; VIOLATED means metric >= threshold.

Correctness means sampled-bucket root-known-state agreement after that
bucket's protocol events, not continuous-time safety or physical latency.

The read-only loader checks the complete control/alpha grid, schemas,
accounting, current input/code/test hashes, C0 event hashes, matching
completion/failure/run records, and the independently executed validation
receipt **before any scientific table/chart is presented**. It does not rerun
tests or replay monitors. Substantive behavioral/saved/replay verification belongs
to `phase3_test.py`; a saved pass flag alone is not independent proof.

In [ ]:
bundle = load_phase3_analysis_results(ROOT, OUTPUT_DIR)
report = bundle["report"]
controls_frame = bundle["controls"]
alpha_frame = bundle["alphas"]
size_frame = bundle["cluster_sizes"]
control_size_frame = bundle["control_cluster_sizes"]
snapshots = bundle["snapshots"]
equivalence = bundle["equivalence"]
revised = bundle["revised"]
historical = bundle["historical"]
config = bundle["config"]
controls = controls_frame.to_dict(orient="records")
alphas = alpha_frame.to_dict(orient="records")
minima = report["alpha_minima"]
ALPHA_LABELS = tuple(f"{step / 20:.2f}" for step in range(1, 20))
figures = {}
chart_series = {}
input_hashes = bundle["input_hashes"]
display(pd.DataFrame([{
    "run_id": report["run_id"],
    "saved_directory": str(OUTPUT_DIR),
    "sensor_ids": ", ".join(report["experiment"]["sensor_ids"]),
    "buckets": report["experiment"]["bucket_count"],
    "threshold_full_precision": repr(report["experiment"]["threshold"]),
    "validation_checks": ", ".join(report["validation"]["checks"]),
    "executed_test_count": report["validation"]["test_count"],
    "implementation_sha256": report["provenance"]["phase3_implementation_sha256"],
    "test_sha256": json.dumps(report["provenance"]["test_sha256"], sort_keys=True),
}]).T.rename(columns={0: "saved value"}))

## 2. Existing method and source explanations

Clustering uses discounted signed reference-deviation histories, normalized by
the largest absolute history; all-zero histories have a deterministic singleton
fallback. Candidate merges preserve signed cancellation, use the existing
alpha-dependent quality, and select the earliest minimum deterministically.
Histories and raw-reading moments are distinct persistent quantities.
The source-specific FMIC/SMIC interpretation below is retained from the former
implementation notebook; it is explanation, not a second monitoring engine.

### First-moment provenance and recurrence

Source: [Clustering for Monitoring Distributed Data Streams, section 1.4, printed page 10 / PDF page 14](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=14), the discounted vector set immediately before the moment-based node constraints. For observation index `m >= 1`, the source set is `V'_n(m) = {2^(-(m-i)) * v_n(t_i) : i=1,...,m}` and `mu_n` is its arithmetic mean. Section 1.5.2 identifies FMIC as using only the first moment.

Therefore `S_n(m) = sum(i=1..m, 2^(-(m-i)) * v_n(t_i)) = 0.5*S_n(m-1) + v_n(t_m)` and `mu_n(m) = S_n(m)/m`, with `S_n(0)=0`. The recurrence is an algebraic implementation of that source set, not a separately quoted source equation. Divide by the observation count `m`, not by the sum of discount weights; this is not an exponentially weighted average.

Project sampling convention: initialization `t0` installs references but is not a moment sample. Every subsequent one-minute raw reading contributes, including synchronization rows. Neither `S_n` nor `m` resets at synchronization, and `m` is not a synchronization count. Before the first sample, the implementation exposes zero moments to the separately labeled project allocation fallback.

Hand calculation for post-initialization readings A=`[2,4,6]`, B=`[-4,2,3]`:

| m | Discounted A observations | S_A | mu_A | Discounted B observations | S_B | mu_B |
|---|---|---|---|---|---|---|
| 1 | [2] | 2 | 2 | [-4] | -4 | -4 |
| 2 | [1,4] | 5 | 5/2 | [-2,2] | 0 | 0 |
| 3 | [0.5,2,6] | 8.5 | 8.5/3 | [-1,1,3] | 3 | 1 |

The separate clustering history is `h_n(t)=0.5*h_n(t-1)+(v_n(t)-reference_n)`, not a raw-reading moment and not divided by `m`. FMIC applies `abs(mu_n)` only when computing weights; signed first moments are preserved. The hand-calculated regression is `FirstMomentTests.test_discounted_arithmetic_mean_is_not_history_or_ema` in [the consolidated tests](phase3_test.py); synchronization persistence is checked separately by `test_history_and_first_moment_persist_across_sync`.

### SMIC second-moment provenance and stable recurrence

The **rendered** source section 1.4 (printed p.10/PDF 14) defines `sigma_n^2` as a sum of squared deviations without a divisor, but prints an inconsistent index `i=0..m`. [Appendix 1, printed p.22/PDF 26](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=26) resolves this: `sigma_n^2 = sum(i=1..m, (v'_n(t_i)-mu_n)^2)`. We use those `m` observations, with **no division by m or m-1**. Despite the source's term 'central second moment', this quantity is not conventional population/sample variance.

[Appendix 1, printed p.24/PDF 28](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=28) gives a stable update: with `k` previous observations, old mean `mu`, and new raw value `x`, `sigma_new^2 = 0.25*sigma_old^2 + k/(k+1)*(x-0.5*mu)^2`. It scales every previous vector by 1/2 and appends x. This reuses the unchanged first-moment update, keeps one common observation count, and never resets at root synchronization. Both terms are nonnegative, so no numerical cancellation correction or negative-value clamp is used; nonfinite or negative states fail explicitly.

Hand example A from above: at m=1, sigma^2=0 and W=2; at m=2, squared deviations `[9/4,9/4]` sum to `9/2`, giving `W=5/2+sqrt((1/2)*(9/2))=4`; at m=3, `[49/9,25/36,361/36]` sum to `97/6`, giving `W=17/6+sqrt(97/9)`. `SecondMomentTests.test_source_discounted_squared_deviations_and_weights_by_hand` verifies both sensors at every step.

SMIC uses `W_n=abs(mu_n)+sqrt(((m-1)/m)*sigma_n^2)`, then the same normalized node and mean coordinator allowances as FMIC. At m=0, zero moments and equal weights are an explicit project initialization convention; at m=1 the central sum and conservative contribution are zero. The approved project all-zero fallback applies only if the total weight is zero; an individual zero weight otherwise stays zero.

`FMICMonitor` retains its compatibility name but is the single lifecycle for `weight_method='FMIC'` or `'SMIC'`. Moment/weight payloads accompany existing sensor-to-root synchronization responses and allowances accompany existing bundled updates. No extra exchanges or internal-computation messages are charged. References, clustering, signed cancellation, escalation, next-bucket activation, and two-sided correctness policy are shared. Each method generates its own schedule and writes distinct compact outputs; FMIC schemas remain unchanged.

### Shared protocol and transmission units

The root is separate from sensor/coordinators. The coordinator is the
lexicographically greatest cluster member. Local absolute-deviation equality
triggers (`>=`); the coordinator takes absolute value **after** averaging signed
member deviations. The root allowance is the absolute distance between the
synchronized mean and P90, on either side of the threshold. At exact P90, zero
allowances cause checks even for unchanged readings on the following bucket.

Triggering non-coordinators report once; the coordinator requests/responses only
from peers not already reporting. Coordinator self-readings are free. Every
violating coordinator sends one root alert; multiple alerts cause one collection
per bucket. Alerts contain no raw readings, so root requests/retrieves all six
states and sends one bundled update per sensor. Requests, responses and updates
are distinct actual transmissions, not a sensor-as-root broadcast approximation.
Moments fit existing responses and allowances existing updates; internal
computation and scalar payload fields are not additional messages.

References, partition and allowances stay fixed until synchronization; the
synchronization reading updates history against entry references before
installing state for the next bucket. Global means use `math.fsum`. Centralized
truth is an offline oracle, never a hidden communication trigger. Continuous
US-3.24 snapshots allow only the established 1e-12 tolerance; triggers, states,
partitions, message counts and schedules remain exact.

## 3. Comparators and C0 equivalence

Keep the historical **signed-margin** Phase 2 result (38,087 messages; 2,107
synchronizations) separate from the revised **two-sided** Phase 2/C0 comparator
(9,159 messages; 467 synchronizations). The centralized 119,904 transmissions
are a different reference again. Historical percentages must not be relabeled
as savings against revised Phase 2.

C0 fixes equal allowances and singleton clusters, ignores alpha/moment values
for decisions, and is verified per bucket against revised Phase 2. C1 and C2
also fix singleton clusters but use FMIC and SMIC weights respectively. Controls
have no alpha trajectory. C0 may have been genuinely regenerated or reused only
with current provenance; the saved run explicitly records which occurred.

In [ ]:
comparator_table = pd.DataFrame([
    {"reference": "Historical signed-margin Phase 2 (not the primary denominator)",
     "actual_transmissions": historical["total_messages"],
     "root_synchronizations": historical["resynchronization_count"]},
    {"reference": "Revised two-sided Phase 2 / C0 (primary comparator)",
     "actual_transmissions": revised["total_messages"],
     "root_synchronizations": revised["resynchronization_count"]},
    {"reference": "Centralized transmission reference",
     "actual_transmissions": report["comparisons"]["centralized_reference"]["total_messages"],
     "root_synchronizations": pd.NA},
])
display(comparator_table)
display(pd.DataFrame([{
    "C0_reused": report["prerequisite_gates"]["C0_reused_from_US-3.24"],
    "per_bucket_equivalence": equivalence["comparison"]["passed"],
    "compared_buckets": equivalence["contract"]["bucket_count"],
    "messages": equivalence["message_mapping"]["phase3_total_messages"],
    "root_synchronizations": equivalence["message_mapping"]["root_synchronizations_including_initialization"],
    "event_sha256": equivalence["outputs"]["events_sha256"],
}]))

## RQ1. Current baseline, controls, and prespecified alpha 0.10

The validated revised Phase 2 and its per-bucket-equivalent C0 result are one reference, not independent experiments. C1/C2 are fixed-weighting singleton controls; FMIC/SMIC at alpha 0.10 are the preselected clustered cases. The centralized row is a distinct one-way sensor-report convention. Message differences are method minus comparator; each percentage names its denominator. Root synchronizations are reported separately from transmissions. The saved-only loader above validates current provenance, category/pathway accounting, correctness, and C0 equivalence before these results are displayed.

In [ ]:
rq1_controls = controls_frame.set_index("config_id")
rq1_alpha10 = alpha_frame.loc[alpha_frame["alpha_label"].eq("0.10")].set_index("algorithm")
if set(rq1_controls.index) != {"C0", "C1", "C2"} or set(rq1_alpha10.index) != {"FMIC", "SMIC"}:
    raise ValueError("RQ1 requires the three validated controls and both alpha=0.10 clustered results.")

rq1_saved_rows = controls_frame.to_dict(orient="records") + alpha_frame.to_dict(orient="records")
if len(rq1_saved_rows) != 41 or any(
    row["correctness_status"] != "passed"
    or int(row["false_safe_count"]) != 0
    or int(row["false_violated_count"]) != 0
    for row in rq1_saved_rows
):
    raise ValueError("RQ1 requires all 41 saved Phase 3 rows to pass both correctness-mismatch checks.")

phase2_c0_messages = int(revised["total_messages"])
centralized_messages = int(report["comparisons"]["centralized_reference"]["total_messages"])
if (
    not equivalence["comparison"]["passed"]
    or int(equivalence["contract"]["bucket_count"]) != 19984
    or phase2_c0_messages != int(rq1_controls.loc["C0", "total_messages"])
    or centralized_messages != 119904
):
    raise ValueError("RQ1 Phase 2/C0 equivalence or fixed-reference gates failed.")

def rq1_comparison_row(method, role, result, matched_control=None):
    total = int(result["total_messages"])
    matched_total = int(matched_control["total_messages"]) if matched_control is not None else None
    return {
        "method": method,
        "role": role,
        "total_actual_transmissions": total,
        "difference_vs_Phase2/C0_messages": total - phase2_c0_messages,
        "difference_vs_Phase2/C0_percent_(denominator=Phase2/C0_messages)": round(
            100 * (total - phase2_c0_messages) / phase2_c0_messages, 1
        ),
        "matched_singleton_control": matched_control.name if matched_control is not None else pd.NA,
        "difference_vs_matched_control_messages": total - matched_total if matched_total is not None else pd.NA,
        "difference_vs_matched_control_percent_(denominator=matched_control_messages)": (
            round(100 * (total - matched_total) / matched_total, 1)
            if matched_total is not None else pd.NA
        ),
        "reduction_vs_one-way_centralized_percent_(denominator=119904)": round(
            100 * (1 - total / centralized_messages), 1
        ),
        "root_synchronizations_including_initialization": int(result["root_synchronization_count"]),
        "false_safe_count": int(result["false_safe_count"]),
        "false_violated_count": int(result["false_violated_count"]),
    }

rq1_rows = [
    rq1_comparison_row(
        "Revised Phase 2 / C0",
        "Equivalent two-sided reference; counted once",
        rq1_controls.loc["C0"],
    ),
    rq1_comparison_row("C1", "FMIC-weighted singleton control", rq1_controls.loc["C1"]),
    rq1_comparison_row("C2", "SMIC-weighted singleton control", rq1_controls.loc["C2"]),
    rq1_comparison_row(
        "FMIC alpha=0.10",
        "Prespecified full clustered method",
        rq1_alpha10.loc["FMIC"],
        rq1_controls.loc["C1"],
    ),
    rq1_comparison_row(
        "SMIC alpha=0.10",
        "Prespecified full clustered method",
        rq1_alpha10.loc["SMIC"],
        rq1_controls.loc["C2"],
    ),
    {
        "method": "One-way centralized reference",
        "role": "Different protocol convention; one report per sensor per bucket",
        "total_actual_transmissions": centralized_messages,
        "difference_vs_Phase2/C0_messages": pd.NA,
        "difference_vs_Phase2/C0_percent_(denominator=Phase2/C0_messages)": pd.NA,
        "matched_singleton_control": pd.NA,
        "difference_vs_matched_control_messages": pd.NA,
        "difference_vs_matched_control_percent_(denominator=matched_control_messages)": pd.NA,
        "reduction_vs_one-way_centralized_percent_(denominator=119904)": pd.NA,
        "root_synchronizations_including_initialization": pd.NA,
        "false_safe_count": pd.NA,
        "false_violated_count": pd.NA,
    },
]
rq1_table = pd.DataFrame(rq1_rows)
display(rq1_table)

rq1_labels = ["Phase 2/C0", "C1", "C2", "FMIC 0.10", "SMIC 0.10"]
rq1_counts = [int(row["total_actual_transmissions"]) for row in rq1_rows[:-1]]
fig, axes = plt.subplots(
    1, 2, figsize=(11, 4.8), constrained_layout=True,
    gridspec_kw={"width_ratios": [4, 1]},
)
small_bars = axes[0].bar(rq1_labels, rq1_counts, color=["#4c78a8", "#9ecae9", "#9ecae9", "#f58518", "#e45756"])
axes[0].set_title("Validated Phase 2 / Phase 3 methods")
axes[0].set_ylabel("Total actual transmissions")
axes[0].tick_params(axis="x", rotation=18)
axes[0].set_ylim(0, max(rq1_counts) * 1.15)
axes[0].grid(axis="y", alpha=0.2)
for bar, value in zip(small_bars, rq1_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value, f"{value:,}", ha="center", va="bottom", fontsize=8)
central_bar = axes[1].bar(["One-way\ncentralized"], [centralized_messages], color="#777777")
axes[1].set_title("Different protocol\nconvention")
axes[1].set_ylim(0, centralized_messages * 1.12)
axes[1].grid(axis="y", alpha=0.2)
for bar in central_bar:
    axes[1].text(bar.get_x() + bar.get_width() / 2, centralized_messages, f"{centralized_messages:,}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Figure 4.1 — Total actual-transmission comparison")
figures["figure_4_1_rq1_baseline.png"] = fig
chart_series["figure_4_1_rq1_baseline.png"] = {
    "phase2_phase3_methods": {"method": rq1_labels, "messages": rq1_counts},
    "one_way_centralized_reference": {"messages": centralized_messages},
}
plt.show()

fmic_messages = int(rq1_alpha10.loc["FMIC", "total_messages"])
smic_messages = int(rq1_alpha10.loc["SMIC", "total_messages"])
c1_messages = int(rq1_controls.loc["C1", "total_messages"])
c2_messages = int(rq1_controls.loc["C2", "total_messages"])
fmic_control_delta = fmic_messages - c1_messages
smic_control_delta = smic_messages - c2_messages
fmic_phase2_delta = fmic_messages - phase2_c0_messages
smic_phase2_delta = smic_messages - phase2_c0_messages
rq1_findings = (
    f"**RQ1 finding.** At the prespecified alpha 0.10, FMIC records {fmic_messages:,} transmissions, "
    f"{fmic_control_delta:+,} ({100 * fmic_control_delta / c1_messages:+.1f}% of C1's {c1_messages:,}) "
    f"versus its matched singleton; SMIC records {smic_messages:,}, "
    f"{smic_control_delta:+,} ({100 * smic_control_delta / c2_messages:+.1f}% of C2's {c2_messages:,}) "
    f"versus its matched singleton. Relative to the single revised Phase 2/C0 reference "
    f"({phase2_c0_messages:,}), the differences are {fmic_phase2_delta:+,} "
    f"({100 * fmic_phase2_delta / phase2_c0_messages:+.1f}% of Phase 2/C0) for FMIC and "
    f"{smic_phase2_delta:+,} ({100 * smic_phase2_delta / phase2_c0_messages:+.1f}% of Phase 2/C0) for SMIC. "
    f"Thus alpha 0.10 does not outperform C1/C2 or Phase 2/C0 on transmission count. "
    f"The clustered runs each have {int(rq1_alpha10.loc['FMIC', 'root_synchronization_count']):,} root synchronizations, "
    f"versus {int(rq1_controls.loc['C1', 'root_synchronization_count']):,}/{int(rq1_controls.loc['C2', 'root_synchronization_count']):,} "
    f"for C1/C2 and {int(rq1_controls.loc['C0', 'root_synchronization_count']):,} for Phase 2/C0; these are operations, not transmissions. "
    f"Both clustered cases use {100 * (1 - fmic_messages / centralized_messages):.1f}% (FMIC) and "
    f"{100 * (1 - smic_messages / centralized_messages):.1f}% (SMIC) fewer transmissions than the {centralized_messages:,}-report "
    f"one-way centralized reference, which is not a like-for-like protocol comparison. "
    f"All 41 saved Phase 3 rows pass with zero false-safe and zero false-violated states; "
    f"the current Phase 2/C0 proof establishes per-bucket equivalence across {int(equivalence['contract']['bucket_count']):,} buckets. "
    "The exploratory alpha-grid minima are separate and do not replace this prespecified result."
)
display(Markdown(rq1_findings))

## 4. Weighting without clustering; matched clustering contrasts

C0 minus C1/C2 isolates weighting under the shared fixed-singleton policy.
C1 versus clustered FMIC and C2 versus clustered SMIC are the matched clustering
contrasts conditional on weighting. FMIC minus SMIC at equal alpha compares
complete methods, not second moments in isolation from their changed schedules.
The prespecified alpha is **0.10**. Tested-grid minima are exploratory, in-sample,
and must remain labeled separately.

In [ ]:
comparison_table = pd.DataFrame(report["configuration_comparison_table"])
comparison_columns = [
    "comparison_scope", "config_id", "alpha_label", "total_messages",
    "root_synchronization_count", "coordinator_check_count",
    "checks_resolved_without_escalation", "checks_escalated",
    "centralized_reduction_percent",
]
display(comparison_table[comparison_columns])
display(pd.json_normalize(report["comparisons"]["weighting_effects_without_clustering"],
                          sep=" / ").T.rename(columns={0: "saved value"}))
matched_rows = []
for method in ("FMIC", "SMIC"):
    for label in sorted({"0.10", *minima[method]["alpha_labels"]}):
        matched_rows.append({
            "algorithm": method, "alpha_label": label,
            "scope": "prespecified" if label == "0.10" else "exploratory in-sample minimum",
            **report["comparisons"]["clustering_effects_conditional_on_weighting"][method][label],
        })
display(pd.DataFrame(matched_rows))
display(pd.DataFrame(report["comparisons"]["full_fmic_minus_smic_at_same_alpha"]).T)

## US-4.03. Allowance allocation without clustering

C0, C1, and C2 are the validated six-singleton controls; this isolates allowance allocation from clustering. Pairwise message percentages use the first-named control's transmissions as the denominator. A positive first-minus-second difference means the second control uses fewer transmissions or operations. Root synchronization counts are operations, while synchronization requests, responses, state updates, and coordinator escalations are separately counted transmissions.

The allocation examples below are independently selected from the current C1/C2 synchronization snapshots. They are not paired events or evidence that the two methods synchronize at matching times. C0 has no saved Phase 3 snapshot, so no C0 sensor allowances are shown. The saved-only loader has already verified the current run manifest and snapshot file hash.


In [ ]:
allowance_control_ids = ("C0", "C1", "C2")
allowance_controls = controls_frame.set_index("config_id", verify_integrity=True)
if set(allowance_controls.index) != set(allowance_control_ids):
    raise ValueError("US-4.03 requires exactly the validated C0/C1/C2 controls.")
allowance_controls = allowance_controls.loc[list(allowance_control_ids)]
if not allowance_controls["alpha_label"].eq("not_applicable").all():
    raise ValueError("Singleton controls must not have alpha-dependent rows.")
control_sensor_count = len(report["experiment"]["sensor_ids"])
if not allowance_controls["cluster_count_mean_per_sync_event"].eq(control_sensor_count).all():
    raise ValueError("C0/C1/C2 must remain six-singleton controls.")
if not allowance_controls["singleton_partition_event_share_percent"].eq(100).all():
    raise ValueError("C0/C1/C2 include a non-singleton control partition.")

message_category_columns = (
    "node_trigger_messages", "cluster_check_requests", "cluster_check_responses",
    "coordinator_root_escalations", "root_sync_requests", "root_sync_responses",
    "root_state_updates",
)
pathway_columns = (
    "within_cluster_messages", "coordinator_to_root_messages", "root_wide_messages",
)
for config_id, control in allowance_controls.iterrows():
    if sum(int(control[column]) for column in message_category_columns) != int(control["total_messages"]):
        raise ValueError(f"{config_id} message categories do not reconcile.")
    if sum(int(control[column]) for column in pathway_columns) != int(control["total_messages"]):
        raise ValueError(f"{config_id} pathway totals do not reconcile.")
    if int(control["within_cluster_messages"]) != 0:
        raise ValueError(f"{config_id} singleton control has within-cluster messages.")

allowance_control_summary = allowance_controls.reset_index()[[
    "config_id", "alpha_label", "cluster_count_mean_per_sync_event",
    "singleton_partition_event_share_percent", "total_messages", "root_synchronization_count",
    *message_category_columns, *pathway_columns,
]].copy()
allowance_control_summary["root_sync_transmissions"] = allowance_control_summary[
    ["root_sync_requests", "root_sync_responses", "root_state_updates"]
].sum(axis=1)
display(allowance_control_summary)

allowance_contrasts = []
saved_control_contrasts = report["comparisons"]["weighting_effects_without_clustering"][
    "matched_pathway_and_check_contrasts"
]
for source_id, comparison_id in (("C0", "C1"), ("C0", "C2"), ("C1", "C2")):
    source = allowance_controls.loc[source_id]
    comparison = allowance_controls.loc[comparison_id]
    row = {
        "contrast (source minus comparison)": f"{source_id} minus {comparison_id}",
        "source_transmissions": int(source["total_messages"]),
        "comparison_transmissions": int(comparison["total_messages"]),
        "transmissions saved by comparison": int(source["total_messages"] - comparison["total_messages"]),
        "saved percent (denominator=source transmissions)": round(
            100 * (source["total_messages"] - comparison["total_messages"]) / source["total_messages"], 3
        ),
        "root synchronizations (source minus comparison)": int(
            source["root_synchronization_count"] - comparison["root_synchronization_count"]
        ),
    }
    for column in (
        "coordinator_root_escalations", "root_sync_requests", "root_sync_responses",
        "root_state_updates", *pathway_columns,
    ):
        row[f"{column} (source minus comparison)"] = int(source[column] - comparison[column])
    row["root_sync_transmissions (source minus comparison)"] = int(
        sum(source[column] - comparison[column]
            for column in ("root_sync_requests", "root_sync_responses", "root_state_updates"))
    )
    saved = saved_control_contrasts[f"{source_id}_minus_{comparison_id}"]
    if (
        saved["message_difference"] != row["transmissions saved by comparison"]
        or saved["root_synchronization_difference"] != row["root synchronizations (source minus comparison)"]
        or not math.isclose(
            saved["percent_relative_to_source"],
            row["saved percent (denominator=source transmissions)"],
            rel_tol=1e-3,
            abs_tol=0.001,
        )
    ):
        raise ValueError(f"{source_id}/{comparison_id} does not match the saved control comparison.")
    for column in message_category_columns:
        if saved["message_category_differences"][column] != int(source[column] - comparison[column]):
            raise ValueError(f"{source_id}/{comparison_id} saved category contrast mismatch: {column}.")
    for column in pathway_columns:
        if saved["pathway_message_differences"][column] != int(source[column] - comparison[column]):
            raise ValueError(f"{source_id}/{comparison_id} saved pathway contrast mismatch: {column}.")
    allowance_contrasts.append(row)
allowance_contrast_table = pd.DataFrame(allowance_contrasts)
display(allowance_contrast_table)

singleton_snapshots = snapshots.loc[snapshots["config_id"].isin(("C1", "C2"))].copy()
if (snapshots["config_id"] == "C0").any():
    raise ValueError("C0 must not be presented as having saved Phase 3 sensor-allocation snapshots.")
if set(singleton_snapshots["config_id"]) != {"C1", "C2"} or not singleton_snapshots["alpha_label"].eq("not_applicable").all():
    raise ValueError("US-4.03 snapshots must be limited to alpha-independent C1/C2 controls.")
snapshot_hash_entries = []
snapshot_path = (OUTPUT_DIR / "phase3_snapshots.csv").resolve()
for filename, digest in input_hashes.items():
    candidate = Path(filename)
    if not candidate.is_absolute():
        candidate = ROOT / candidate
    if candidate.resolve() == snapshot_path:
        snapshot_hash_entries.append(digest)
if len(snapshot_hash_entries) != 1:
    raise ValueError("The validated current-run snapshot hash is missing or ambiguous.")

sensor_ids = list(report["experiment"]["sensor_ids"])
def snapshot_mapping(row, column):
    value = row[column]
    if isinstance(value, dict):
        mapping = value
    elif pd.isna(value) or not str(value).strip():
        return {}
    else:
        mapping = json.loads(value)
    if not isinstance(mapping, dict) or set(mapping) != set(sensor_ids):
        raise ValueError(f"Snapshot {row['config_id']} sync {row['synchronization_id']} has an invalid {column} map.")
    return {sensor: float(mapping[sensor]) for sensor in sensor_ids}

def close_mappings(actual, expected):
    return set(actual) == set(expected) and all(
        math.isclose(actual[sensor], expected[sensor], rel_tol=1e-12, abs_tol=1e-12)
        for sensor in expected
    )

validated_snapshot_maps = {}
expected_snapshot_contract = {
    "C1": ("FMIC", "source_first_moment"),
    "C2": ("SMIC", "source_first_plus_second_moment"),
}
for _, snapshot in singleton_snapshots.iterrows():
    config_id = snapshot["config_id"]
    algorithm, source_status = expected_snapshot_contract[config_id]
    sync_id = int(snapshot["synchronization_id"])
    if snapshot["snapshot_type"] != "synchronization" or snapshot["algorithm"] != algorithm:
        raise ValueError(f"{config_id} sync {sync_id} is not its own current algorithm synchronization snapshot.")
    partition = json.loads(snapshot["selected_partition"])
    if len(partition) != len(sensor_ids) or any(len(cluster) != 1 for cluster in partition):
        raise ValueError(f"{config_id} sync {sync_id} is not a six-singleton snapshot.")
    if sorted(sensor for cluster in partition for sensor in cluster) != sorted(sensor_ids):
        raise ValueError(f"{config_id} sync {sync_id} does not cover the current sensor cohort.")
    if int(pd.Timestamp(snapshot["bucket_time_utc"]).timestamp()) != int(snapshot["bucket_epoch"]):
        raise ValueError(f"{config_id} sync {sync_id} timestamp does not match its saved event epoch.")

    first = snapshot_mapping(snapshot, "first_moments")
    raw = snapshot_mapping(snapshot, "raw_weights")
    scaled = snapshot_mapping(snapshot, "scaled_weights")
    node = snapshot_mapping(snapshot, "node_allowances")
    coordinator = snapshot_mapping(snapshot, "coordinator_allowances")
    moment_count = int(snapshot["moment_observation_count"])
    if config_id == "C1":
        expected_raw = {sensor: abs(first[sensor]) for sensor in sensor_ids}
        terms = {}
        second = {}
    else:
        second = snapshot_mapping(snapshot, "second_moments")
        terms = snapshot_mapping(snapshot, "conservative_terms")
        if moment_count > 0:
            expected_terms = {
                sensor: math.sqrt(((moment_count - 1) / moment_count) * second[sensor])
                for sensor in sensor_ids
            }
            if not close_mappings(terms, expected_terms):
                raise ValueError(f"C2 sync {sync_id} does not match the source-defined SMIC contribution.")
            expected_raw = {
                sensor: abs(first[sensor]) + expected_terms[sensor]
                for sensor in sensor_ids
            }
        else:
            expected_raw = dict.fromkeys(sensor_ids, 0.0)
            if any(not math.isclose(first[sensor], 0.0, abs_tol=1e-12) for sensor in sensor_ids):
                raise ValueError(f"C2 sync {sync_id} has unavailable moments but nonzero first moments.")
    if not close_mappings(raw, expected_raw):
        raise ValueError(f"{config_id} sync {sync_id} saved raw weights do not match its source moments.")

    weight_status = snapshot["weight_status"]
    if math.fsum(raw.values()) == 0:
        if weight_status != "project_all_zero_equal_weight_fallback":
            raise ValueError(f"{config_id} sync {sync_id} all-zero weights lack the explicit fallback status.")
        expected_scaled = dict.fromkeys(sensor_ids, 1.0)
    else:
        if weight_status != source_status:
            raise ValueError(f"{config_id} sync {sync_id} has an unexpected source-weight status.")
        total_weight = math.fsum(raw.values())
        expected_scaled = {sensor: len(sensor_ids) * raw[sensor] / total_weight for sensor in sensor_ids}
    if not close_mappings(scaled, expected_scaled):
        raise ValueError(f"{config_id} sync {sync_id} normalized weights do not reconcile.")

    root_allowance = float(snapshot["root_allowance"])
    expected_node = {sensor: scaled[sensor] * root_allowance for sensor in sensor_ids}
    if not close_mappings(node, expected_node) or not math.isclose(
        math.fsum(node.values()), len(sensor_ids) * root_allowance, rel_tol=1e-12, abs_tol=1e-12
    ):
        raise ValueError(f"{config_id} sync {sync_id} node allowances do not sum to six root allowances.")
    if math.isclose(root_allowance, 0.0, abs_tol=1e-12) and any(
        not math.isclose(value, 0.0, abs_tol=1e-12) for value in node.values()
    ):
        raise ValueError(f"{config_id} sync {sync_id} zero root allowance produced a nonzero node allowance.")
    if not close_mappings(coordinator, node):
        raise ValueError(f"{config_id} sync {sync_id} singleton coordinator allowances differ from node allowances.")
    validated_snapshot_maps[(config_id, sync_id)] = {
        "first_moments": first, "second_moments": second, "conservative_terms": terms,
        "raw_weights": raw, "scaled_weights": scaled, "node_allowances": node,
    }

snapshot_status_counts = singleton_snapshots.groupby(["config_id", "weight_status"]).size().unstack(fill_value=0)
display(snapshot_status_counts)

example_rows = []
for config_id, source_status in (("C1", "source_first_moment"), ("C2", "source_first_plus_second_moment")):
    candidates = singleton_snapshots.loc[
        (singleton_snapshots["config_id"] == config_id)
        & (singleton_snapshots["weight_status"] == source_status)
        & (singleton_snapshots["root_allowance"] > 0)
    ].sort_values(["bucket_epoch", "synchronization_id"])
    if candidates.empty:
        raise ValueError(f"No genuine source-weighted positive-allowance {config_id} snapshot is available.")
    event = candidates.iloc[0]
    values = validated_snapshot_maps[(config_id, int(event["synchronization_id"]))]
    for sensor in sensor_ids:
        example_rows.append({
            "control": config_id,
            "algorithm": event["algorithm"],
            "synchronization_id": int(event["synchronization_id"]),
            "actual_event_time_utc": event["bucket_time_utc"],
            "moment_observation_count": int(event["moment_observation_count"]),
            "weight_status": event["weight_status"],
            "sensor": sensor,
            "first_moment_mu_C": values["first_moments"][sensor],
            "second_moment_sum_sigma2_C2": values["second_moments"].get(sensor, pd.NA),
            "SMIC_second_moment_contribution_C": values["conservative_terms"].get(sensor, pd.NA),
            "raw_weight": values["raw_weights"][sensor],
            "normalized_weight (cohort total=6)": values["scaled_weights"][sensor],
            "node_allowance_C": values["node_allowances"][sensor],
            "root_allowance_C": float(event["root_allowance"]),
        })
allowance_snapshot_examples = pd.DataFrame(example_rows)
for column in (
    "first_moment_mu_C", "second_moment_sum_sigma2_C2", "SMIC_second_moment_contribution_C",
    "raw_weight", "normalized_weight (cohort total=6)", "node_allowance_C", "root_allowance_C",
):
    allowance_snapshot_examples[column] = pd.to_numeric(allowance_snapshot_examples[column], errors="coerce").round(6)
display(allowance_snapshot_examples)

snapshot_provenance = pd.DataFrame([{
    "current_run_id": report["run_id"],
    "snapshot_file": str(snapshot_path),
    "validated_snapshot_sha256": snapshot_hash_entries[0],
    "snapshot_rows_validated": len(singleton_snapshots),
    "C1_snapshot_rows": int((singleton_snapshots["config_id"] == "C1").sum()),
    "C2_snapshot_rows": int((singleton_snapshots["config_id"] == "C2").sum()),
    "C0_snapshot_rows": 0,
}])
display(snapshot_provenance)
display(Markdown(
    "**Interpretation.** These three controls differ only in allowance allocation under the shared singleton protocol. "
    "A root synchronization is an operation; its three six-sensor transmission categories are counted separately from "
    "the one-message-per-coordinator root escalation. The selected C1/C2 examples are independently taken from their "
    "own saved synchronization events, not matched schedule events. C0 has no saved Phase 3 snapshot, so no C0 sensor "
    "allowances are inferred. SMIC's source-defined squared-deviation sum is not ordinary sample or population variance."
))


## 5. Existing interpretation and alpha sensitivity

The following calculation is relocated unchanged from the revised runner's
`_interpretation` function and uses **saved rows only**. It retains existing
ranges, direction reversals, matched contrasts, selected runs, pathways and
check proportions; it does not introduce new metrics or run an experiment.

In [ ]:
from typing import Any

def _interpretation(
    controls: list[dict[str, Any]],
    alphas: list[dict[str, Any]],
    revised: dict[str, Any],
    minima: dict[str, Any],
) -> dict[str, Any]:
    by_id = {row["config_id"]: row for row in controls}

    def selected_run(row: dict[str, Any]) -> dict[str, Any]:
        return {
            "alpha_label": row["alpha_label"],
            "total_messages": row["total_messages"],
            "root_synchronization_count": row["root_synchronization_count"],
            "revised_phase2_reduction_percent": row[
                "reduction_vs_revised_phase2_percent"
            ],
            "matched_singleton_message_difference": row[
                "matched_singleton_message_difference"
            ],
            "matched_singleton_reduction_percent": row[
                "matched_singleton_reduction_percent"
            ],
            "matched_singleton_root_synchronization_difference": row[
                "matched_singleton_root_synchronization_difference"
            ],
            "coordinator_check_count": row["coordinator_check_count"],
            "checks_resolved_without_escalation": row[
                "checks_resolved_without_escalation"
            ],
            "checks_escalated": row["checks_escalated"],
            "locally_resolved_check_share_percent": (
                100 * row["checks_resolved_without_escalation"]
                / row["coordinator_check_count"]
                if row["coordinator_check_count"] else None
            ),
            "repeated_coordinator_check_count": row[
                "repeated_coordinator_check_count"
            ],
            "repeated_check_share_percent": row["repeated_check_share_percent"],
            "repeated_checks_resolved_without_escalation": row[
                "repeated_checks_resolved_without_escalation"
            ],
            "repeated_locally_resolved_share_percent": (
                100 * row["repeated_checks_resolved_without_escalation"]
                / row["repeated_coordinator_check_count"]
                if row["repeated_coordinator_check_count"] else None
            ),
            "communication_pathways": {
                name: {
                    "messages": row[name],
                    "share_percent": row[f"{name}_share_percent"],
                }
                for name in (
                    "within_cluster_messages",
                    "coordinator_to_root_messages",
                    "root_wide_messages",
                )
            },
            "mean_cluster_count_per_synchronization": row[
                "cluster_count_mean_per_sync_event"
            ],
            "singleton_partition_event_share_percent": row[
                "singleton_partition_event_share_percent"
            ],
        }

    sensitivity: dict[str, Any] = {}
    for method in ("FMIC", "SMIC"):
        series = sorted(
            (row for row in alphas if row["algorithm"] == method),
            key=lambda row: float(row["alpha_label"]),
        )
        differences = [
            current["total_messages"] - previous["total_messages"]
            for previous, current in zip(series, series[1:])
        ]
        signs = [1 if value > 0 else -1 if value < 0 else 0 for value in differences]
        nonzero_signs = [sign for sign in signs if sign]
        reversals = sum(
            left != right
            for left, right in zip(nonzero_signs, nonzero_signs[1:])
        )
        alpha10 = next(row for row in series if row["alpha_label"] == "0.10")
        control = by_id["C1" if method == "FMIC" else "C2"]
        best_runs = [
            next(
                row for row in series
                if row["alpha_label"] == alpha_label
            )
            for alpha_label in minima[method]["alpha_labels"]
        ]
        sensitivity[method] = {
            "message_total_range": {
                "minimum": min(row["total_messages"] for row in series),
                "maximum": max(row["total_messages"] for row in series),
                "direction_reversals_across_tested_grid": reversals,
            },
            "root_synchronization_range": {
                "minimum": min(row["root_synchronization_count"] for row in series),
                "maximum": max(row["root_synchronization_count"] for row in series),
            },
            "weighting_only_contrast": {
                "C0_minus_weighted_singleton_messages": (
                    by_id["C0"]["total_messages"] - control["total_messages"]
                ),
                "percent_relative_to_C0": (
                    100
                    * (by_id["C0"]["total_messages"] - control["total_messages"])
                    / by_id["C0"]["total_messages"]
                    if by_id["C0"]["total_messages"] else None
                ),
            },
            "prespecified_alpha_0.10": selected_run(alpha10),
            "exploratory_minimum_runs": [
                selected_run(row) for row in best_runs
            ],
        }
    return {
        "message_sensitivity": sensitivity,
        "matched_revised_phase2_messages": revised["total_messages"],
        "communication_by_cluster_size_source": (
            "phase3_cluster_sizes.csv"
        ),
        "interpretive_cautions": [
            "Grid message totals are irregular when direction reversals are present; minima are exploratory and in-sample.",
            "A locally resolved coordinator check is not evidence that a root synchronization was counterfactually avoided.",
            "Persistent local-allowance triggers and new global threshold-state changes are distinct; global correctness is checked against the independent per-bucket oracle.",
            "Repeated checks describe current protocol behavior and are not evidence that any check can safely be skipped.",
            "Message totals do not measure bytes, latency, or bandwidth; no statistical significance or availability robustness is inferred.",
        ],
    }

interpretation = _interpretation(controls, alphas, revised, minima)
display(pd.json_normalize(interpretation["message_sensitivity"], sep=" / ").T.rename(columns={0: "saved-result analysis"}))

### Figure 4.3 and Figure 4.5: Alpha overview charts

These are the existing revised alpha charts. Fixed singleton controls are
benchmarks, not alpha-dependent runs. Fewer root synchronizations do not imply
fewer total transmissions once within-cluster checks are counted. The historical
signed-margin benchmark is **not** the dashed revised comparison below.

In [ ]:
definitions = (
    ("total_messages", "Actual transmissions", "phase3a_us325_messages_vs_alpha.png"),
    ("root_synchronization_count", "Root synchronizations (including initialization)", "phase3a_us325_synchronizations_vs_alpha.png"),
    ("cluster_count_mean_per_sync_event", "Mean clusters per synchronization event", "phase3a_us325_clustering_vs_alpha.png"),
)
figure_titles = {
    "total_messages": "Figure 4.3 - Total actual transmissions versus alpha",
    "root_synchronization_count": "Figure 4.5a - Root synchronization operations versus alpha",
    "cluster_count_mean_per_sync_event": "Figure 4.5b - Mean clusters per synchronization versus alpha",
}
for column, ylabel, name in definitions:
    fig, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
    chart_series[name] = {}
    for method, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
        data = alpha_frame.loc[alpha_frame["algorithm"] == method].sort_values("alpha")
        axis.plot(data["alpha"], data[column], marker="o", color=color, label=method)
        chart_series[name][method] = {"alpha": data["alpha"].tolist(), "values": data[column].tolist()}
    if column == "total_messages":
        axis.axhline(revised["total_messages"], color="#444444", linestyle="--", label="Revised Phase 2/C0 (fixed benchmark)")
        for control in controls:
            if control["config_id"] in ("C1", "C2"):
                axis.axhline(control["total_messages"], color="#777777", linestyle=":", linewidth=1,
                             label=f'{control["config_id"]} singleton benchmark (fixed)')
    axis.set_xlabel("Alpha (full clustered runs only)")
    axis.set_ylabel(ylabel)
    axis.set_xticks([float(label) for label in ALPHA_LABELS])
    axis.grid(True, alpha=0.25)
    axis.legend()
    fig.suptitle(figure_titles[column])
    figures[name] = fig
    plt.show()

## US-4.04. Alpha sensitivity, break-even, and exploratory minimum

Alpha 0.10 remains the prespecified reference. All other tested-grid findings,
including minima and near-minimum bands, are exploratory and in-sample. Each
clustered alpha is compared only with its own fixed singleton weighting control;
positive control-minus-clustered differences mean actual transmissions saved.
Figure 4.4 marks the discrete tested break-even points without interpolation.
The 19-pair table, adjacent-step changes, and status regions use validated current
saved rows only. Root synchronization counts below are operation counts, not
transmissions.

In [ ]:
alpha_control_ids = {"FMIC": "C1", "SMIC": "C2"}
alpha_control_rows = controls_frame.set_index("config_id", verify_integrity=True)
if set(alpha_control_rows.index) != {"C0", "C1", "C2"}:
    raise ValueError("US-4.04 requires exactly the validated C0/C1/C2 controls.")
if len(alpha_frame) != 38:
    raise ValueError("US-4.04 requires the complete 38-row clustered alpha grid.")

alpha_table_rows = []
alpha_step_rows = []
alpha_break_even_rows = []
alpha_near_minimum_rows = []
alpha_order = {label: index for index, label in enumerate(ALPHA_LABELS)}
for method, control_id in alpha_control_ids.items():
    method_rows = alpha_frame.loc[alpha_frame["algorithm"] == method].copy()
    observed_labels = set(method_rows["alpha_label"])
    if len(method_rows) != 19 or observed_labels != set(ALPHA_LABELS):
        raise ValueError(f"{method} must have exactly the 19 canonical tested alpha labels.")
    method_rows["_alpha_order"] = method_rows["alpha_label"].map(alpha_order)
    method_rows = method_rows.sort_values("_alpha_order")
    if method_rows["alpha_label"].tolist() != list(ALPHA_LABELS):
        raise ValueError(f"{method} alpha rows are not in canonical two-decimal order.")
    control = alpha_control_rows.loc[control_id]
    if control["alpha_label"] != "not_applicable":
        raise ValueError(f"{control_id} must be a fixed alpha-independent control.")
    control_messages = int(control["total_messages"])
    saved_minimum = minima[method]
    if saved_minimum["scope"] != "exploratory_in_sample_tested_grid":
        raise ValueError(f"{method} saved minimum is not labeled exploratory and in-sample.")

    method_records = method_rows.to_dict(orient="records")
    status_records = []
    for row in method_records:
        label = row["alpha_label"]
        total = int(row["total_messages"])
        saved = control_messages - total
        saved_percent = 100 * saved / control_messages
        saved_syncs = int(control["root_synchronization_count"]) - int(row["root_synchronization_count"])
        if (
            saved != int(row["matched_singleton_message_difference"])
            or not math.isclose(saved_percent, float(row["matched_singleton_reduction_percent"]), rel_tol=1e-10, abs_tol=1e-10)
            or saved_syncs != int(row["matched_singleton_root_synchronization_difference"])
        ):
            raise ValueError(f"{method} alpha {label} does not match its saved singleton contrast.")
        comparison = report["comparisons"]["clustering_effects_conditional_on_weighting"][method][label]
        if (
            comparison["message_difference"] != saved
            or not math.isclose(float(comparison["percent_relative_to_source"]), saved_percent, rel_tol=1e-10, abs_tol=1e-10)
        ):
            raise ValueError(f"{method} alpha {label} does not match the saved comparison summary.")
        status = "beat" if saved > 0 else "tie" if saved == 0 else "lose"
        status_records.append({"alpha_label": label, "status": status, "saved": saved})
        alpha_table_rows.append({
            "method": method,
            "alpha": label,
            "total actual transmissions": total,
            "matched singleton control": control_id,
            "control transmissions": control_messages,
            "transmissions saved by clustering": saved,
            "percent saved (denominator=control transmissions)": saved_percent,
            "status versus matched control": status,
            "clustered root synchronizations (operations)": int(row["root_synchronization_count"]),
            "control root synchronizations (operations)": int(control["root_synchronization_count"]),
        })

    computed_minimum = int(method_rows["total_messages"].min())
    computed_minimum_labels = [
        label for label, total in zip(method_rows["alpha_label"], method_rows["total_messages"])
        if int(total) == computed_minimum
    ]
    if (
        computed_minimum != int(saved_minimum["minimum_total_messages"])
        or computed_minimum_labels != saved_minimum["alpha_labels"]
    ):
        raise ValueError(f"{method} calculated minimum differs from the validated saved minimum.")
    for band in (0.01, 0.05):
        labels = [
            label for label, total in zip(method_rows["alpha_label"], method_rows["total_messages"])
            if float(total) <= computed_minimum * (1 + band)
        ]
        alpha_near_minimum_rows.append({
            "method": method,
            "exploratory tested-grid minimum transmissions": computed_minimum,
            "minimum alpha labels": ", ".join(computed_minimum_labels),
            "band": f"within {band:.0%} of minimum",
            "near-minimum tested alpha labels": ", ".join(labels),
        })

    statuses = [record["status"] for record in status_records]
    regions = []
    start = 0
    for index in range(1, len(status_records) + 1):
        if index == len(status_records) or statuses[index] != statuses[start]:
            region_labels = [record["alpha_label"] for record in status_records[start:index]]
            region = statuses[start] + " " + region_labels[0]
            if len(region_labels) > 1:
                region += "-" + region_labels[-1]
            regions.append(region)
            start = index
    transitions = [
        f"{previous['alpha_label']} {previous['status']} -> {current['alpha_label']} {current['status']}"
        for previous, current in zip(status_records, status_records[1:])
        if previous["status"] != current["status"]
    ]
    alpha_break_even_rows.append({
        "method": method,
        "tested status regions": "; ".join(regions),
        "tested alphas beating control": ", ".join(record["alpha_label"] for record in status_records if record["status"] == "beat") or "none",
        "tested alphas tying control": ", ".join(record["alpha_label"] for record in status_records if record["status"] == "tie") or "none",
        "tested alphas losing to control": ", ".join(record["alpha_label"] for record in status_records if record["status"] == "lose") or "none",
        "first tested break-even crossing": transitions[0] if transitions else "none in tested grid",
        "subsequent tested reversals": "; ".join(transitions[1:]) if len(transitions) > 1 else "none",
    })
    for previous, current in zip(method_records, method_records[1:]):
        previous_total = int(previous["total_messages"])
        current_total = int(current["total_messages"])
        alpha_step_rows.append({
            "method": method,
            "adjacent tested alpha step": f"{previous['alpha_label']} -> {current['alpha_label']}",
            "change in total transmissions (later - earlier)": current_total - previous_total,
            "percent change (denominator=earlier total)": 100 * (current_total - previous_total) / previous_total,
        })

alpha_sensitivity_table = pd.DataFrame(alpha_table_rows)
display(alpha_sensitivity_table)
alpha_break_even_table = pd.DataFrame(alpha_break_even_rows)
display(alpha_break_even_table)
alpha_near_minimum_table = pd.DataFrame(alpha_near_minimum_rows)
display(alpha_near_minimum_table)
alpha_step_change_table = pd.DataFrame(alpha_step_rows)
display(alpha_step_change_table)

alpha_10_findings = []
for method, control_id in alpha_control_ids.items():
    alpha_10 = alpha_sensitivity_table.loc[
        (alpha_sensitivity_table["method"] == method) & (alpha_sensitivity_table["alpha"] == "0.10")
    ].iloc[0]
    control = alpha_control_rows.loc[control_id]
    clustered = alpha_frame.loc[
        (alpha_frame["algorithm"] == method) & (alpha_frame["alpha_label"] == "0.10")
    ].iloc[0]
    local_extra = int(clustered["within_cluster_messages"] - control["within_cluster_messages"])
    root_wide_saved = int(control["root_wide_messages"] - clustered["root_wide_messages"])
    coordinator_saved = int(control["coordinator_to_root_messages"] - clustered["coordinator_to_root_messages"])
    if int(alpha_10["transmissions saved by clustering"]) != -local_extra + root_wide_saved + coordinator_saved:
        raise ValueError(f"{method} alpha 0.10 pathway changes do not reconcile to total transmissions.")
    transmission_difference = int(alpha_10["transmissions saved by clustering"])
    transmission_result = (
        f"uses {-transmission_difference:,} more actual transmissions than {control_id}"
        if transmission_difference < 0 else
        f"saves {transmission_difference:,} actual transmissions versus {control_id}"
        if transmission_difference > 0 else
        f"ties {control_id} on actual transmissions"
    )
    sync_difference = int(alpha_10["control root synchronizations (operations)"]) - int(alpha_10["clustered root synchronizations (operations)"])
    synchronization_result = (
        f"{sync_difference:,} fewer root synchronization operations"
        if sync_difference > 0 else
        f"{-sync_difference:,} more root synchronization operations"
        if sync_difference < 0 else
        "the same number of root synchronization operations"
    )
    alpha_10_findings.append(
        f"{method} at prespecified alpha 0.10 {transmission_result}, despite {synchronization_result}; "
        f"within-cluster traffic is {local_extra:+,} versus the control."
    )
display(Markdown(
    "**US-4.04 interpretation.** " + " ".join(alpha_10_findings) + " "
    "Fewer root synchronization operations are not automatically fewer transmissions: the increase in local coordination can exceed root-wide savings. "
    "US-4.05 examines the seven transmission categories behind this tradeoff. "
    "The prespecified 0.10 result is distinct from the exploratory, in-sample tested-grid minimum; no interpolation, curve fitting, significance test, or claim about untested alphas is made."
))

figure_44_name = "figure_4_4_matched_control_break_even.png"
fig, axis = plt.subplots(figsize=(9, 5.2), constrained_layout=True)
axis.axhline(0, color="#222222", linewidth=1.8, label="Break-even: equal total transmissions")
axis.axvline(0.10, color="#777777", linestyle="--", linewidth=1.2, label="Prespecified alpha = 0.10")
for method, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
    plotted = alpha_sensitivity_table.loc[alpha_sensitivity_table["method"] == method]
    x_values = [float(label) for label in plotted["alpha"]]
    y_values = plotted["transmissions saved by clustering"].tolist()
    axis.plot(x_values, y_values, marker="o", color=color, label=f"{method} vs {alpha_control_ids[method]}")
    minimum_labels = minima[method]["alpha_labels"]
    minimum_rows = plotted.loc[plotted["alpha"].isin(minimum_labels)]
    axis.scatter(
        [float(label) for label in minimum_rows["alpha"]],
        minimum_rows["transmissions saved by clustering"],
        color=color, marker="*", s=150, edgecolor="black", linewidth=0.5,
        label=f"{method} exploratory minimum",
        zorder=4,
    )
    chart_series.setdefault(figure_44_name, {})[method] = {
        "alpha": x_values,
        "transmissions_saved_vs_matched_control": y_values,
        "matched_control": alpha_control_ids[method],
        "exploratory_minimum_alpha_labels": minimum_labels,
    }
axis.set_xlabel("Alpha (tested values only)")
axis.set_ylabel("Actual transmissions saved by clustering\nrelative to matched weighting-only control")
axis.set_xticks([float(label) for label in ALPHA_LABELS])
axis.set_title("Figure 4.4 - Clustering break-even by matched weighting method")
axis.grid(True, alpha=0.25)
axis.legend(ncol=2, fontsize=8)
figures[figure_44_name] = fig
plt.show()

## 6. Communication composition, local checks, and repetition

The existing three pathways are within-cluster (trigger reports plus peer
requests/responses), coordinator-to-root alerts, and root-wide
requests/responses/updates. Their totals reconcile to actual transmissions.
The singleton bar chart remains alpha-independent.

In [ ]:
group_columns = ("within_cluster_messages", "coordinator_to_root_messages", "root_wide_messages")
colors = ("#4c78a8", "#f58518", "#54a24b")
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True, sharey=True)
composition_name = "phase3a_us325_communication_composition.png"
chart_series[composition_name] = {}
for axis, method in zip(axes, ("FMIC", "SMIC")):
    data = alpha_frame.loc[alpha_frame["algorithm"] == method].sort_values("alpha")
    axis.stackplot(data["alpha"], *(data[column] for column in group_columns),
                   labels=("Within-cluster", "Coordinator-to-root", "Root-wide"), colors=colors, alpha=0.85)
    chart_series[composition_name][method] = {column: data[column].tolist() for column in ("alpha", *group_columns)}
    axis.set_title(method)
    axis.set_xlabel("Alpha")
    axis.set_xticks([float(label) for label in ALPHA_LABELS])
    axis.grid(True, alpha=0.2)
axes[0].set_ylabel("Actual transmissions by pathway")
axes[1].legend(loc="upper right")
figures[composition_name] = fig
plt.show()

fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
singleton_name = "phase3a_us325_singleton_communication_composition.png"
labels = controls_frame["config_id"].tolist()
bottom = [0] * len(controls)
for column, label, color in zip(group_columns, ("Within-cluster", "Coordinator-to-root", "Root-wide"), colors):
    values = controls_frame[column].tolist()
    axis.bar(labels, values, bottom=bottom, label=label, color=color)
    bottom = [prior + value for prior, value in zip(bottom, values)]
axis.set_xlabel("Alpha-independent singleton control")
axis.set_ylabel("Actual transmissions")
axis.set_title("Communication composition by control")
axis.legend()
axis.grid(axis="y", alpha=0.2)
chart_series[singleton_name] = {column: controls_frame[column].tolist() for column in ("config_id", *group_columns)}
figures[singleton_name] = fig
plt.show()

### Existing check and cluster-size diagnostics

A locally resolved check is **not** a counterfactual root synchronization
avoided. Repeated checks mean adjacent buckets, unchanged membership, and no
intervening synchronization. They describe the current policy, not evidence that
checks can safely be skipped. Persistent allowance triggers and new global state
changes are distinct. Checks, internal computation, and messages are not
interchangeable units.

In [ ]:
check_columns = [
    "config_id", "alpha_label", "coordinator_check_count", "checks_resolved_without_escalation",
    "checks_escalated", "repeated_coordinator_check_count",
    "repeated_checks_resolved_without_escalation", "repeated_check_share_percent",
    "singleton_partition_event_share_percent",
]
display(comparison_table.loc[comparison_table["config_id"] != "revised_phase2", check_columns])
display(control_size_frame)
display(size_frame)
display(snapshots[["config_id", "alpha_label", "synchronization_id", "bucket_time_utc",
                   "selected_partition", "global_average", "p90_threshold",
                   "root_allowance", "moment_observation_count"]])

## 7. Preserved historical sweep charts (separate comparator)

The original US-3.22 exports used the signed-margin Phase 2 benchmark. They are
historical presentation, not current validation or evidence against revised
Phase 2. To retain their existing content without conflating denominators, the
following **opt-in** section reads the complete saved historical sweep; it never
launches it. Missing historical inputs fail if requested. No automatic fallback
from missing revised results is permitted.

In [ ]:
SHOW_HISTORICAL = os.environ.get("PHASE3_SHOW_HISTORICAL", "0") == "1"
if SHOW_HISTORICAL:
    historical_bundle = load_phase3_analysis_results(ROOT, OUTPUT_DIR, include_historical=True)
    history_rows = historical_bundle["historical_alphas"]
    input_hashes.update(historical_bundle["input_hashes"])
    historical_definitions = (
        ("total_messages", "Actual transmissions", "phase3a_messages_vs_alpha.png"),
        ("root_synchronization_count", "Root synchronizations (including initialization)", "phase3a_synchronizations_vs_alpha.png"),
        ("cluster_count_mean_per_sync_event", "Mean clusters per synchronization event", "phase3a_clustering_vs_alpha.png"),
    )
    for column, ylabel, name in historical_definitions:
        fig, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        chart_series[name] = {}
        for method, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
            data = history_rows.loc[history_rows["algorithm"] == method].sort_values("alpha")
            axis.plot(data["alpha"], data[column], marker="o", color=color, label=f"{method} (historical correctness passed)")
            chart_series[name][method] = {"alpha": data["alpha"].tolist(), "values": data[column].tolist()}
        if column == "total_messages":
            axis.axhline(historical["total_messages"], color="#444444", linestyle="--",
                         label="Historical signed-margin Phase 2 (fixed benchmark)")
        axis.set_xlabel("Alpha (tested grid: 0.05 to 0.95 by 0.05)")
        axis.set_ylabel(ylabel)
        axis.set_title(f"Historical Phase 3A fixed A/C/D/E/F/G cohort: {ylabel} vs alpha")
        axis.set_xticks([float(label) for label in ALPHA_LABELS])
        axis.grid(True, alpha=0.25)
        axis.legend()
        figures[name] = fig
        plt.show()
else:
    display(Markdown("Historical charts not requested. Set `PHASE3_SHOW_HISTORICAL=1` before kernel start to render the preserved three-chart view."))

## 8. Existing limitations, analytical gaps, and export policy

These are controlled within-dataset contrasts, not general causal estimates.
The grid minimum is exploratory and in-sample, not a universal optimum.
Messages do not measure bytes, latency, or bandwidth. No statistical significance,
out-of-sample benefit, general method ranking, or availability robustness is
inferred from this fixed six-sensor temperature cohort.

**Deferred analytical gaps (not filled here):** independent/out-of-sample tuning,
statistical uncertainty, byte/latency measurements, availability-varying cohorts,
and counterfactual protocol/check suppression require separate research stories.
This notebook adds no experiments, performance metrics, or claims.

The five revised PNG exports previously came from production; their content is
now inline here. The three historical PNGs retain their original identities and
separate benchmark when requested. Production no longer generates the revised
PNGs or detailed interpretive narrative. Historical files are preserved, not
deleted or silently rewritten. Optional exports must use a **new, separate**
directory and the same historical filenames. Default execution does not export.

In [ ]:
for caution in interpretation["interpretive_cautions"]:
    display(Markdown(f"- {caution}"))
for limitation in report["interpretation_limits"]:
    display(Markdown(f"- {limitation}"))

EXPORT_DIR = os.environ.get("PHASE3_ANALYSIS_EXPORT_DIR")
if EXPORT_DIR:
    export_path = Path(EXPORT_DIR)
    if not export_path.is_absolute():
        export_path = ROOT / export_path
    export_path = export_path.resolve()
    protected = [OUTPUT_DIR, ROOT / "artifacts" / "phase3_operational", ROOT / "artifacts" / "phase3_archive", ROOT / "scripts", ROOT / "config", ROOT / "Data", ROOT / ".git"]
    if (export_path == ROOT or ROOT.is_relative_to(export_path) or export_path == ROOT / "artifacts"
        or any(export_path == path or export_path.is_relative_to(path) for path in protected)
        or (export_path.is_relative_to(ROOT / "artifacts") and
            any(part.startswith(("phase3_baseline_", "phase3_extraction_", "phase3_production_", "phase3_testing_", "phase3_analysis_"))
                for part in export_path.relative_to(ROOT / "artifacts").parts))):
        raise ValueError("Analysis exports require a new directory outside saved outputs, source and migration evidence.")
    export_path.mkdir(parents=True, exist_ok=False)
    for name, figure in figures.items():
        figure.savefig(export_path / name, dpi=160)
    print(f"Explicit PNG exports: {export_path}")

for filename, expected in input_hashes.items():
    with Path(filename).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise RuntimeError(f"Analysis input changed during execution: {filename}")
print(f"Read-only analysis complete: {len(controls)} controls, {len(alphas)} clustered runs, {len(figures)} inline figures.")